In [24]:
from google.colab import files
uploaded = files.upload()

In [25]:
!pip install faiss-cpu

In [26]:
import faiss
import pickle

# load index
index = faiss.read_index("/content/pubmedqa_index_flatl2.faiss")

# load mapping
with open("/content/chunk_mapping.pkl", "rb") as f:
    text_chunks = pickle.load(f)

In [27]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
def retrieve(query, k=3):
    query_vector = model.encode([query])
    D, I = index.search(query_vector, k)
    return [text_chunks['question'].iloc[i] for i in I[0]]

In [29]:
query = "What are the symptoms of diabetes?"

results = retrieve(query)

context = "\n".join(results)

print("Context:\n", context)

Context:
 abnormalities in glucose homeostasis in acromegaly does the prevalence of glucose intolerance depend on the level of activity of the disease and the duration of the symptoms
can glycaemic variability as calculated from blood glucose self monitoring predict the development of complications in type 1 diabetes over a decade
difficult diagnosis in a 17 year old patient type 1 diabetes


In [30]:
!pip install python-dotenv

In [31]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [32]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful medical assistant. Answer only based on the given context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
)

print(response.choices[0].message.content)

However, I want to clarify that this patient may have Type 1 diabetes or possibly diabetes related to acromegaly, but based on the context, I'll provide general symptoms of diabetes.

Common symptoms of diabetes can include:

1. Increased thirst and hunger: This is due to high blood sugar levels.
2. Frequent urination: This is a result of the body trying to flush out excess glucose in the urine.
3. Fatigue: High blood sugar levels can lead to feeling weak or tired.
4. Blurred vision: High blood sugar levels can cause the lens in the eye to swell, leading to blurred vision.
5. Cuts or wounds that are slow to heal: High blood sugar levels can affect the body's ability to heal wounds.
6. Tingling or numbness in the hands and feet: High blood sugar levels can damage nerves.
7. Repeated skin, gum, or bladder infections: High blood sugar levels can weaken the body's immune system.

These symptoms can be similar for both Type 1 and Type 2 diabetes, as well as diabetes related to conditions li

In [33]:
DISCLAIMER = "\n\n⚠️ This system is for educational purposes only."

answer = response.choices[0].message.content + DISCLAIMER
print(answer)

However, I want to clarify that this patient may have Type 1 diabetes or possibly diabetes related to acromegaly, but based on the context, I'll provide general symptoms of diabetes.

Common symptoms of diabetes can include:

1. Increased thirst and hunger: This is due to high blood sugar levels.
2. Frequent urination: This is a result of the body trying to flush out excess glucose in the urine.
3. Fatigue: High blood sugar levels can lead to feeling weak or tired.
4. Blurred vision: High blood sugar levels can cause the lens in the eye to swell, leading to blurred vision.
5. Cuts or wounds that are slow to heal: High blood sugar levels can affect the body's ability to heal wounds.
6. Tingling or numbness in the hands and feet: High blood sugar levels can damage nerves.
7. Repeated skin, gum, or bladder infections: High blood sugar levels can weaken the body's immune system.

These symptoms can be similar for both Type 1 and Type 2 diabetes, as well as diabetes related to conditions li

In [34]:
log = {
    "question": query,
    "context": context,
    "answer": answer
}

print(log)

{'question': 'What are the symptoms of diabetes?', 'context': 'abnormalities in glucose homeostasis in acromegaly does the prevalence of glucose intolerance depend on the level of activity of the disease and the duration of the symptoms\ncan glycaemic variability as calculated from blood glucose self monitoring predict the development of complications in type 1 diabetes over a decade\ndifficult diagnosis in a 17 year old patient type 1 diabetes', 'answer': "However, I want to clarify that this patient may have Type 1 diabetes or possibly diabetes related to acromegaly, but based on the context, I'll provide general symptoms of diabetes.\n\nCommon symptoms of diabetes can include:\n\n1. Increased thirst and hunger: This is due to high blood sugar levels.\n2. Frequent urination: This is a result of the body trying to flush out excess glucose in the urine.\n3. Fatigue: High blood sugar levels can lead to feeling weak or tired.\n4. Blurred vision: High blood sugar levels can cause the lens